# Cloud Provider Analytics — Pipeline completo

```text
Landing → Bronze → Silver → Gold → Serving (AstraDB)
```

Orquestación end-to-end del MVP (parcial 2). La lógica vive en `src/jobs/`; este notebook ejecuta cada capa y muestra evidencias.

**Prerrequisitos:** dataset en `datalake/landing/`, keyspace `cloud_analytics` en consola Astra, token + bundle configurados.

In [ ]:
# Setup (Colab o local)
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pyspark cassandra-driver
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/cloud-provider-analytics")
    # os.environ["DATA_ROOT"] = "/content/datalake"
    # os.environ["ASTRA_DB_APPLICATION_TOKEN"] = "AstraCS:..."
    # os.environ["ASTRA_DB_SECURE_BUNDLE_PATH"] = "/content/secure-connect-cloud-analytics.zip"
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

from src.config import (
    BRONZE,
    CASSANDRA_KEYSPACE,
    CHECKPOINTS,
    CQL_DIR,
    DATA_ROOT,
    GOLD,
    LANDING,
    QUARANTINE,
    SILVER,
)
from src.cassandra.client import is_astra_configured
from src.jobs.bronze_streaming import (
    USAGE_EVENTS_BRONZE_PATH,
    USAGE_EVENTS_CHECKPOINT_PATH,
    USAGE_EVENTS_LANDING_GLOB,
)
from src.jobs.gold_batch import ORG_DAILY_USAGE_BY_SERVICE
from src.jobs.silver_batch import USAGE_EVENTS_QUARANTINE, USAGE_EVENTS_SILVER
from src.schemas.bronze_streaming import WATERMARK_DELAY

print(f"PROJECT_ROOT:      {PROJECT_ROOT}")
print(f"DATA_ROOT:         {DATA_ROOT}")
print(f"Astra configured:  {is_astra_configured()}")
print(f"KEYSPACE:          {CASSANDRA_KEYSPACE}")

In [ ]:
spark = (
    SparkSession.builder.appName("cloud-provider-analytics")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Batch Bronze — 3 maestros CSV

Ingesta de `customers_orgs`, `users` y `billing_monthly` → Parquet tipificado con `ingest_ts`, `source_file` y dedupe.

In [ ]:
from src.jobs.bronze_batch import run_batch_bronze, validate_bronze_uniqueness

batch_results = run_batch_bronze(spark)
display(pd.DataFrame(batch_results)[
    ["dataset_name", "raw_count", "deduped_count", "removed_duplicates", "written_count", "bronze_path"]
])

In [ ]:
for dataset in ["customers_orgs", "users", "billing_monthly"]:
    path = f"{BRONZE}/{dataset}"
    print(f"\n=== {dataset} ===")
    df = spark.read.parquet(path)
    df.printSchema()
    df.select(df.columns[:6]).show(5, truncate=False)

display(pd.DataFrame(validate_bronze_uniqueness(spark)))

## 2. Streaming Bronze — usage_events

Structured Streaming desde `usage_events_stream/*.jsonl` con watermark, dedupe `event_id`, late data y checkpoint.

> Watermark `60 days` para replay del landing estático; producción usa `10 minutes` (`STREAMING_WATERMARK`).

In [ ]:
from src.jobs.bronze_streaming import run_streaming_bronze, validate_bronze_streaming

print(f"LANDING glob: {USAGE_EVENTS_LANDING_GLOB}")
print(f"CHECKPOINT:   {USAGE_EVENTS_CHECKPOINT_PATH}")
print(f"WATERMARK:    {WATERMARK_DELAY}")

streaming_result = run_streaming_bronze(spark, reset_state=True)
display(pd.DataFrame([streaming_result]))

In [ ]:
df = spark.read.parquet(USAGE_EVENTS_BRONZE_PATH)
df.printSchema()
df.select(
    "event_id", "event_ts", "service", "value", "schema_version",
    "carbon_kg", "genai_tokens", "source_file", "is_late_arrival"
).show(5, truncate=False)

display(pd.DataFrame([validate_bronze_streaming(spark)]))

## 3. Silver — eventos + customers_orgs

Joins, features (`daily_cost_usd`, `requests`, `genai_tokens`, `carbon_kg`), 3 reglas de calidad y quarantine.

In [ ]:
from src.jobs.silver_batch import run_silver, validate_silver

silver_results = run_silver(spark)
events = silver_results[1]

display(pd.DataFrame([{
    "bronze": events["raw_count"],
    "silver_valid": events["valid_count"],
    "quarantine": events["quarantine_count"],
    "cost_anomalies_flagged": events["cost_anomalies_flagged"],
}]))
events["quarantine_sample"]

In [ ]:
df = spark.read.parquet(USAGE_EVENTS_SILVER)
df.select(
    "event_id", "org_name", "usage_date", "service",
    "daily_cost_usd", "requests", "genai_tokens", "carbon_kg", "is_cost_anomaly"
).show(5, truncate=False)

display(pd.DataFrame([validate_silver(spark)]))

## 4. Gold — org_daily_usage_by_service

Mart FinOps con grano `(org_id, usage_date, service)` y métricas agregadas.

In [ ]:
from src.jobs.gold_batch import run_gold, validate_gold

gold_results = run_gold(spark)
display(pd.DataFrame(gold_results))

In [ ]:
df = spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
df.printSchema()
df.orderBy(df.total_daily_cost_usd.desc()).show(10, truncate=False)

display(pd.DataFrame([validate_gold(spark)]))

## 5. Serving — AstraDB

Carga del mart Gold y consultas #1 y #2. Requiere keyspace `cloud_analytics` en consola Astra + token + bundle.

In [ ]:
from pyspark.sql import functions as F
from src.jobs.serving_cassandra import run_serving

if not is_astra_configured():
    print("Skipping serving: set ASTRA_DB_APPLICATION_TOKEN and ASTRA_DB_SECURE_BUNDLE_PATH.")
else:
    top_org = (
        spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
        .groupBy("org_id")
        .agg(F.sum("total_daily_cost_usd").alias("cost"))
        .orderBy(F.desc("cost"))
        .first()
    )
    org_id = top_org["org_id"] if top_org else "org_rixa11dp"
    result = run_serving(spark, org_id=org_id)
    print("LOAD:", result["load"])
    display(pd.DataFrame(result["query1_sample"]))
    display(pd.DataFrame(result["query2_top"]))